# Этап 2. Время и полнота измерений

Таблица считывается из репозитория **один раз**. Здесь оцениваются также минуты, для которых в файле вообще нет строки. Пропуски не восстанавливаются интерполяцией.

## Имена переменных

- `source` — файл в репозитории; `measurements` — все его измерения в `pandas`.
- `minutes` — полная минутная сетка от первой до последней даты.
- `output_dir` — папка результатов выбранного файла.
- `coverage_table` — полнота активной мощности по месяцам.
- `run_table` — непрерывные последовательности отсутствующих измерений.
- `expected_in_record` — число ожидаемых минут **внутри фактического периода записи**.
- `expected_calendar` — число минут целого календарного месяца, включая дни до начала или после конца файла.
- `valid_power` — число минут с пригодной активной мощностью.
- `coverage_record_pct` — доля пригодных минут в пределах периода записи, в процентах.
- `coverage_calendar_pct` — доля пригодных минут во всём календарном месяце, в процентах.
- `missing_or_invalid` — минуты с отсутствующей/непригодной мощностью.
- `crosses_month` — пересекает ли непрерывный пропуск границу месяца.

In [ ]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

## Расчёт полноты и последовательностей пропусков

Сначала `regular_minutes()` добавляет в **расчётную** таблицу отсутствующие временные метки. Обратите внимание: добавляются только строки со значениями `NaN`, не измеренные значения мощности.

`monthly_coverage()` не предполагает, что каждый файл содержит полный месяц. `missing_runs()` ищет непрерывные пропуски на единой временной сетке, поэтому пропуск между двумя соседними строками не теряется.

In [ ]:
from ipd_analysis import monthly_coverage, missing_runs

minutes = regular_minutes(measurements)
coverage_table = monthly_coverage(minutes)
run_table = missing_runs(minutes)
display(coverage_table)
display(run_table.head(10))
save_csv(coverage_table, output_dir / "02_monthly_coverage.csv")
save_csv(run_table, output_dir / "02_missing_runs.csv")
print("Всего минут без корректной мощности:", int(coverage_table["missing_or_invalid"].sum()))